# Huấn Luyện VAE Trực Tiếp Từ VS Code Qua Extension Google Colab

> **Cách kết nối Kernel trong VS Code:**
> 1. Ở góc trên cùng bên phải của file này trong VS Code, bấm vào **Select Kernel** (hoặc `Ctrl+Shift+P` -> gõ `Notebook: Select Notebook Kernel`).
> 2. Chọn **Colab** -> Chọn **Auto Connect** hoặc tài khoản Google của bạn.
> 3. Trình duyệt sẽ mở popup xác thực Google. Sau khi cấp quyền, VS Code sẽ kết nối trực tiếp với GPU của Colab!
> 4. Bấm nút **Run Cell (▶)** trên từng ô bên dưới để chạy trực tiếp trên GPU cloud.

## 1. Kiểm tra phần cứng GPU của Colab

In [ ]:
!nvidia-smi

## 2. Đồng bộ mã nguồn dự án vào Colab VM

Chọn 1 trong 2 cách sau tùy theo cách bạn lưu trữ code:

### Cách 2A (Khuyên Dùng): Clone trực tiếp từ GitHub repo của nhóm
> *Lưu ý:* Hãy nhớ chạy `git push` từ terminal máy bạn trước để các file mới nhất (`models/vae/`, `train_vae.py`) có trên GitHub.

In [ ]:
import os

# Clone hoặc kéo code mới nhất từ GitHub
if not os.path.exists('/content/DL_ImgGenModel'):
    !git clone https://github.com/lucaspham102/DL_ImgGenModel.git /content/DL_ImgGenModel

%cd /content/DL_ImgGenModel
!git pull origin main
!ls -la

### Cách 2B: Mount Google Drive (Nếu bạn lưu trữ project trên Drive)

In [ ]:
# Bỏ comment nếu bạn muốn chạy từ thư mục lưu trên Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/DL_ImgGenModel
# !ls -la

## 3. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -r requirements.txt
!pip install -q wandb

## 4. Đăng nhập Weights & Biases (W&B)

In [ ]:
import wandb
# Dán API key từ https://wandb.ai/authorize khi được hỏi:
wandb.login()

## 5. Chạy Kiểm Tra Nhanh (Sanity Check - 2 Epochs Fashion-MNIST)

In [ ]:
!python models/vae/train.py \
    --dataset fashion_mnist \
    --subset 2000 \
    --epochs 2 \
    --batch-size 64 \
    --backend wandb

## 6. Huấn luyện chính thức VAE trên CelebA (Khuôn mặt)

In [ ]:
!python models/vae/train.py \
    --dataset celeba \
    --epochs 30 \
    --batch-size 128 \
    --lr 0.0005 \
    --latent-dim 128 \
    --kl-weight 1.0 \
    --backend wandb

## 7. Khôi phục huấn luyện nếu bị ngắt quãng (Resume)

In [ ]:
!python models/vae/train.py \
    --dataset celeba \
    --epochs 30 \
    --batch-size 128 \
    --resume checkpoints/vae_celeba_last.pt \
    --backend wandb

## 8. Đánh giá mô hình VAE (FID, Diversity, Speed)

In [ ]:
!python evaluate.py \
    --model vae \
    --dataset celeba \
    --ckpt checkpoints/vae_celeba_best.pt \
    --num-samples 500